# Tutorial: Hydra orchestration

## Introduction

This notebook creates real temporary configuration and input files, composes them with Hydra, and dispatches small independent pipeline runs. It also creates a reusable server-specific Slurm profile and renders its script without submitting a job. Every computation uses the public Python wrapper.

### Library abstraction

Hydra resolves `_target_` constructors and configuration groups. `plan_configuration` freezes a model-free `ExecutionPlan`, containing a schema and `PlannedRun` entries. `Orchestrator` saves that plan and delegates launch behavior to an `ExecutionBackend`. The run's memory/SQLite storage backend is a separate setting. `LocalBackend` runs sequentially, `SubprocessBackend` uses isolated Python processes, and `SlurmBackend` returns a script path.

### Assumptions and notation

Use the installed CPU environment and bundled HydrAMP weights; no APEX weights or peptide dataset are needed. Fixed peptide strings and seed 7 make the examples deterministic. `input.mode=individual` produces independent jobs; `batch` produces one run. Each plan selects one launch backend for its jobs. Synthetic files and results are created in a temporary directory and removed at the end.

### Cells

Create a complete configuration directory rather than requiring edits to installed package files. The example includes YAML defaults, a CSV, a FASTA and a server-owned shell template.

In [1]:
# USAGE: Create the tutorial workspace and artificial inputs.
from pathlib import Path
from tempfile import TemporaryDirectory
import json
from IPython.display import display
from pep_compass.orchestrator.configuration import compose_configuration
from pep_compass.orchestrator.planning import plan_configuration, save_plan, load_plan
from pep_compass.orchestrator.runner import Orchestrator
from pep_compass.optimization.saving import AnalysisReader

artifacts = TemporaryDirectory(prefix="pep-compass-orchestrator-")
root = Path(artifacts.name)
conf = root / "conf"
(conf / "pipeline").mkdir(parents=True)
(conf / "backend").mkdir()
inputs_path = root / "peptides.csv"
inputs_path.write_text('sequence\nAA\nCC\n', encoding="utf-8")
fasta_path = root / "peptides.fasta"
fasta_path.write_text('>first\nAA\n>second\nCC\n', encoding="utf-8")
print(inputs_path.read_text())
print(fasta_path.read_text())

/home/max/repositories/pep-compass/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sequence
AA
CC

>first
AA
>second
CC



## Configurations use the public constructors

### Relation between components

The pipeline YAML below builds the same objects as `Pipeline([...], autoencoder=ae.HydrAMP(...))`. Constructors store settings; planning validates them without calling `build()`. Workers later reconstruct the frozen schema, initialize resources and call `Pipeline.run()`. Defaults supply omitted constructor parameters.

### Cells

Write a root YAML and three interchangeable launch groups. Resource paths are explicit so each example is independent of the notebook's current directory.

In [2]:
# USAGE: Build a complete, self-contained Hydra configuration group tree.
(conf / "config.yaml").write_text(f"""defaults:
  - pipeline: small
  - backend: local
  - _self_
input:
  sequences: null
  path: {inputs_path.as_posix()}
  mode: individual
  column: sequence
runner:
  execution:
    seed: 7
    device: cpu
  storage:
    backend: sqlite
    metrics_level: basic
orchestrator:
  name: local-example
  output_dir: {root.as_posix()}/results
  skip_completed: false
  continue_on_error: false
plugins: []
""", encoding="utf-8")
(conf / "pipeline" / "small.yaml").write_text("""_target_: pep_compass.Pipeline
autoencoder:
  _target_: pep_compass.autoencoder.HydrAMP
  jacobian_mode: approx
steps:
  - _target_: pep_compass.steps.filters.CandidateSubset
    count: 2
  - _target_: pep_compass.steps.oracles.Hydrophobicity
""", encoding="utf-8")
(conf / "backend" / "local.yaml").write_text(
    '_target_: pep_compass.orchestrator.backends.local.LocalBackend\n', encoding="utf-8")
(conf / "backend" / "subprocess.yaml").write_text(
    '_target_: pep_compass.orchestrator.backends.subprocess.SubprocessBackend\nmax_workers: 2\n', encoding="utf-8")
configuration = compose_configuration(config_dir=conf)
plan = plan_configuration(configuration, working_directory=root)
print("Independent jobs:", len(plan.entries))
assert len(plan.entries) == 2
print(json.dumps(plan.schema.to_dict(), indent=2))

Independent jobs: 2
{
  "steps": [
    {
      "target": "pep_compass.steps.filters.strategies.direct.structural.candidate_subset.CandidateSubset",
      "parameters": {
        "count": 2,
        "mode": "random",
        "score_field": null
      },
      "node_id": "node_d4f2ac130dc0451bbac9933a27ac69ae",
      "name": null,
      "children": {}
    },
    {
      "target": "pep_compass.steps.oracles.strategies.hydrophobicity.component.Hydrophobicity",
      "parameters": {
        "batch_size": null,
        "parallelize": false,
        "num_workers": null,
        "evaluation_budget": null,
        "force_isolation": false,
        "evaluation_batch_size": null,
        "scale": "eisenberg"
      },
      "node_id": "node_f7be500a20da4f16b226d94e90b0e6c1",
      "name": null,
      "children": {}
    }
  ],
  "autoencoder": {
    "_target_": "pep_compass.autoencoder.strategies.hydramp.resource.HydrAMP",
    "model": "article_25",
    "device": "cpu",
    "jacobian_mode": "approx

## Run, save and validate completed jobs

### Data contract

A frozen plan stores constructor snapshots, resolved inputs/settings, code identity and backend parameters. Each job gets a separate run directory. The orchestrator records launch status outside the run bundle; computation results still use `settings.json` plus `results.sqlite`. `skip_completed` validates an existing completed result before skipping it; it does not resume an interrupted graph.

### Cells

Run the local jobs, open each result through the common reader, reload the frozen plan and demonstrate validated skipping.

In [3]:
# USAGE: Dispatch real jobs through Pipeline.run(), then read their retained evaluations.
orchestrator = Orchestrator()
records = orchestrator.run(plan)
assert [record.status for record in records] == ["completed", "completed"]
for record in records:
    with AnalysisReader(record.path) as reader:
        display(reader.get_archive())
        assert len(reader.get_archive()) == 1

# USAGE: Save/load the same plan; launch policy does not replace the computation.
frozen_plan, plan_path = save_plan(plan)
restored_plan = load_plan(plan_path)
assert restored_plan.to_dict() == frozen_plan.to_dict()
from dataclasses import replace
skip_plan = replace(plan, options=replace(plan.options, skip_completed=True))
skipped = orchestrator.run(skip_plan)
assert all(record.status == "skipped" for record in skipped)
print([record.status for record in skipped])

,evaluation_id,objective,sequence,score,direction,evaluation_index,candidate_id,execution_id,iteration_id
0,1,hydrophobicity,AA,0.62,maximize,0,0,4,None


,evaluation_id,objective,sequence,score,direction,evaluation_index,candidate_id,execution_id,iteration_id
0,1,hydrophobicity,CC,0.29,maximize,0,0,4,None


['skipped', 'skipped']


## Overrides and isolated processes

### Configuration contract

An override can replace a group or a constructor parameter. This cell chooses `backend=subprocess`, replaces the input file with FASTA, and switches to a single batch. The pipeline definition remains the same. Independent processes own their model objects and run state; graph-level `Parallel` is a different composition mechanism.

### Cells

Run one isolated worker and read its two evaluated input sequences. A separate experiment name keeps output ownership unambiguous.

In [4]:
# USAGE: Compose a backend group and parameter overrides from the same YAML files.
worker_config = compose_configuration(config_dir=conf, overrides=[
    "backend=subprocess", "backend.max_workers=2",
    f"input.path={fasta_path.as_posix()}", "input.mode=batch",
    "pipeline.steps.0.count=2", "orchestrator.name=subprocess-example",
])
worker_plan = plan_configuration(worker_config, working_directory=root)
worker_records = orchestrator.run(worker_plan)
assert len(worker_records) == 1 and worker_records[0].status == "completed"
with AnalysisReader(worker_records[0].path) as reader:
    display(reader.get_archive())
    assert set(reader.get_archive().sequence) == {"AA", "CC"}

,evaluation_id,objective,sequence,score,direction,evaluation_index,candidate_id,execution_id,iteration_id
0,1,hydrophobicity,AA,0.62,maximize,0,0,4,None
1,2,hydrophobicity,CC,0.29,maximize,1,1,4,None


## Reusable server profiles

### Backend contract

A profile can select an existing `SlurmBackend` with a server-owned template and resource values. A new server variant needs a new YAML profile, not a registry entry or another backend class. `resolve_parameters` reads and freezes template text during planning. The backend checks required placeholders when rendering the script; current code does not certify Bash syntax or cluster resource availability during planning.

The template must include `${command}` and `${array_max}`; `${directives}` and `${working_directory}` insert the backend's directives and resolved directory. Site setup belongs in the shell template. Slurm execution here generates a file; scheduler submission remains a separate action.

### Cells

Create an artificial `slurm_demo_gpu` profile and shell template, select it by name, and validate the generated script locally without calling `sbatch`.

In [5]:
# USAGE: Define site setup once, then choose it through backend=slurm_demo_gpu.
site_template = root / "demo_gpu.sh"
site_template.write_text("""#!/bin/bash
#SBATCH --array=0-${array_max}
${directives}
set -euo pipefail
# REMARK: Replace this demonstration setup with the site's module/container setup.
export PEP_COMPASS_TUTORIAL_SITE=demo_gpu
cd ${working_directory}
${command}
""", encoding="utf-8")
(conf / "backend" / "slurm_demo_gpu.yaml").write_text(f"""_target_: pep_compass.orchestrator.backends.slurm.SlurmBackend
template_path: {site_template.as_posix()}
settings:
  partition: demo_gpu
  time: "00:05:00"
  memory: 4G
  cpus_per_task: 2
  gpus: 1
""", encoding="utf-8")
slurm_config = compose_configuration(config_dir=conf, overrides=[
    "backend=slurm_demo_gpu", "orchestrator.name=slurm-example",
])
slurm_plan = plan_configuration(slurm_config, working_directory=root)
assert slurm_plan.backend["template_path"] is None
assert "PEP_COMPASS_TUTORIAL_SITE" in slurm_plan.backend["template"]
# REMARK: The rendered script uses the frozen template, even if its source file is removed.
site_template.unlink()
script_path = orchestrator.run(slurm_plan)
print(script_path.read_text())
assert "SLURM_ARRAY_TASK_ID" in script_path.read_text()
assert "#SBATCH --partition=demo_gpu" in script_path.read_text()
import subprocess
subprocess.run(["bash", "-n", str(script_path)], check=True)
# REMARK: This notebook performs Bash syntax validation explicitly; the backend does not.

#!/bin/bash
#SBATCH --array=0-1
#SBATCH --partition=demo_gpu
#SBATCH --time=00:05:00
#SBATCH --mem=4G
#SBATCH --cpus-per-task=2
#SBATCH --gpus=1
set -euo pipefail
# REMARK: Replace this demonstration setup with the site's module/container setup.
export PEP_COMPASS_TUTORIAL_SITE=demo_gpu
cd /tmp/pep-compass-orchestrator-_fafcv36
/home/max/repositories/pep-compass/.venv/bin/python -m pep_compass.orchestrator.cli worker /tmp/pep-compass-orchestrator-_fafcv36/results/slurm-example/plans/fa44dd17b3228c6741d481faee90ca454379baf0681fed7101b9739d1bca1a2b.json --index "${SLURM_ARRAY_TASK_ID:?}"



CompletedProcess(args=['bash', '-n', '/tmp/pep-compass-orchestrator-_fafcv36/results/slurm-example/plans/fa44dd17b3228c6741d481faee90ca454379baf0681fed7101b9739d1bca1a2b.slurm.sh'], returncode=0)

## CLI and result ownership

The same configuration directory can be used with the CLI's `--config-path` and `--config-name` options. `plan` prints a model-free plan; `run` dispatches the selected backend. Workers consume a persisted plan and one job index. The server's configured Python environment must be able to import the recorded library version and access its model files.

### Cells

Print commands pointing at the actual generated files. These commands are examples, not automatically submitted jobs. Result readers are already closed before temporary cleanup.

In [6]:
# USAGE: Print commands for the generated configuration directory.
import shlex
print("pep-compass plan --config-path", shlex.quote(str(conf)), "--config-name config backend=slurm_demo_gpu")
print("pep-compass run --config-path", shlex.quote(str(conf)), "--config-name config backend=subprocess orchestrator.name=cli-example")
print("Generated scheduler script:", script_path)
# REMARK: Temporary example files stop existing after this cleanup; keep a chosen directory to reuse them.
artifacts.cleanup()

pep-compass plan --config-path /tmp/pep-compass-orchestrator-_fafcv36/conf --config-name config backend=slurm_demo_gpu
pep-compass run --config-path /tmp/pep-compass-orchestrator-_fafcv36/conf --config-name config backend=subprocess orchestrator.name=cli-example
Generated scheduler script: /tmp/pep-compass-orchestrator-_fafcv36/results/slurm-example/plans/fa44dd17b3228c6741d481faee90ca454379baf0681fed7101b9739d1bca1a2b.slurm.sh
